# Exploratory Analysis of a Simulated Biomedical Cohort

**Portfolio demonstration — entirely synthetic records. No patient data are used.**

## Objective
Demonstrate a reproducible workflow for data validation, descriptive statistics, group comparison, and cautious interpretation of a small biomedical-style dataset.

## Dataset
The notebook generates fictional records with age, a binary group label, and a continuous biomarker value. These are invented values and do not represent a disease, real population, diagnostic threshold, or treatment effect.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(7)
n_per_group = 60
group = np.repeat(["Group A", "Group B"], n_per_group)
age = rng.integers(18, 75, size=2*n_per_group)
biomarker = np.concatenate([
    rng.normal(12.0, 2.5, n_per_group),
    rng.normal(13.1, 2.8, n_per_group)
])
df = pd.DataFrame({"group": group, "age_years": age, "biomarker_arb_units": biomarker})
df.head()

In [ ]:
# Basic data-quality checks
print("Shape:", df.shape)
print("\nMissing values:\n", df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("\nData types:\n", df.dtypes)

assert df["group"].isin(["Group A", "Group B"]).all()
assert df["age_years"].between(18, 100).all()
assert df["biomarker_arb_units"].notna().all()

In [ ]:
summary = df.groupby("group")["biomarker_arb_units"].agg(
    ["count", "mean", "std", "median", "min", "max"]
)
summary

In [ ]:
a = df.loc[df.group == "Group A", "biomarker_arb_units"]
b = df.loc[df.group == "Group B", "biomarker_arb_units"]

# Welch's t-test does not assume equal population variances.
test = stats.ttest_ind(a, b, equal_var=False)
mean_difference = float(b.mean() - a.mean())

print(f"Mean difference (B - A): {mean_difference:.3f} arbitrary units")
print(f"Welch t statistic: {test.statistic:.3f}")
print(f"Two-sided p-value: {test.pvalue:.4f}")

In [ ]:
plt.figure(figsize=(7, 5))
plt.boxplot([a, b], tick_labels=["Group A", "Group B"])
plt.ylabel("Biomarker (arbitrary units)")
plt.title("Simulated biomarker distributions")
plt.grid(axis="y", alpha=0.25)
plt.show()

## Interpretation

Use the computed summaries and test output after running the notebook. A small p-value, if observed, would not establish causation, clinical importance, or generalizability. A non-significant result would not prove that groups are equivalent.

## Limitations
- All observations are simulated.
- No confounder adjustment, missing-data mechanism, sampling design, or clinical context is represented.
- The analysis is illustrative; real biomedical studies require a prespecified analysis plan, appropriate ethics and privacy safeguards, and domain-specific interpretation.

## Reproducibility
A fixed random seed (`7`) makes the synthetic dataset reproducible. Required packages: NumPy, pandas, SciPy, and Matplotlib.